# Coupang Internal Brand Review Analysis
Analyzes Coupang product reviews for the company's own diaper product lines.
Extracts satisfaction/dissatisfaction keywords, generates property scores and summaries,
and flags risk-related mentions using GPT-4.

**Pipeline sections:**
1. Keyword extraction (satisfaction vs dissatisfaction)
2. Property rating + weekly summary
3. Risk keyword detection

**Run:** Trigger by updating `start` and `end` dates at the top.

In [ ]:
# pip install googletrans==4.0.0rc1 snowflake-connector-python snowflake-sqlalchemy==1.5.1 openai

In [ ]:
import os
import ast
import time
import pandas as pd
import openai
import snowflake.connector
from datetime import datetime, timedelta
from googletrans import Translator
from snowflake.sqlalchemy import URL
from sqlalchemy import create_engine
from snowflake.connector.pandas_tools import pd_writer

# Date range for this batch run (update before each run)
start = (datetime.now() - timedelta(days=15)).strftime('%Y-%m-%d')
end   = (datetime.now() - timedelta(days=9)).strftime('%Y-%m-%d')
print(f'Processing: {start} to {end}')

In [ ]:
# ── DATABASE CONNECTION ──────────────────────────────────────────────────
# Set environment variables before running:
#   SNOWFLAKE_USER, SNOWFLAKE_PASSWORD, SNOWFLAKE_ACCOUNT
#   SNOWFLAKE_DATABASE, SNOWFLAKE_SCHEMA, SNOWFLAKE_WAREHOUSE
#   OPENAI_API_KEY

con = snowflake.connector.connect(
    user=os.environ['SNOWFLAKE_USER'],
    password=os.environ['SNOWFLAKE_PASSWORD'],
    account=os.environ['SNOWFLAKE_ACCOUNT'],
    database=os.environ['SNOWFLAKE_DATABASE'],
    schema=os.environ['SNOWFLAKE_SCHEMA'],
    warehouse=os.environ['SNOWFLAKE_WAREHOUSE']
)
cur = con.cursor()

## Part 1: Keyword Extraction
GPT-4 extracts top 3 satisfaction and dissatisfaction keywords per batch of reviews.
Results are translated to English via Google Translate.

In [ ]:
# ── LOAD REVIEW DATA ─────────────────────────────────────────────────────

review_query = f"""
SELECT DATE, max(BRAND), max(SUBBRAND), REVIEW
FROM REVIEW_TABLE
WHERE DATE >= '{start}' AND DATE <= '{end}'
GROUP BY DATE, REVIEW, PROPERTY;
"""
review = pd.DataFrame(cur.execute(review_query))
review.columns = ['DATE', 'BRAND', 'SUBBRAND', 'REVIEW']

In [ ]:
# ── GPT-4 KEYWORD EXTRACTION ─────────────────────────────────────────────

openai.api_key = os.environ['OPENAI_API_KEY']
GPT_MODEL = 'gpt-4o'

# Product sub-brands to analyze
TARGET_SUBBRANDS = [
    'MaxDry', 'CleanBebe', 'MagicPants', 'Bosongbosong', 'NatureBambu',
    'SwimmingPants', 'NatureMade', 'GoodNight', 'NatureMadeOrganic',
    'MagicSummer', 'GreenFinger', 'NatureSummer', 'PureCotton'
]

def extract_keywords(reviews, brand):
    """Extract top 3 satisfaction and dissatisfaction keywords from a batch of reviews."""
    prompt = (
        f'Reviews: {reviews} '
        'Prompt: The above are diaper reviews. Excluding delivery and sizing, '
        'identify 3 keywords each for what customers were satisfied and dissatisfied with. '
        'Output format: {"satisfied": keywords, "unsatisfied": keywords}. No other output.'
    )
    messages = [
        {'role': 'system', 'content': f'You are a marketer for the {brand} diaper brand.'},
        {'role': 'user', 'content': prompt}
    ]
    response = openai.ChatCompletion.create(model=GPT_MODEL, messages=messages)
    return response.choices[0].message.content

In [ ]:
# ── BATCH KEYWORD EXTRACTION ─────────────────────────────────────────────

col_sub, col_word, col_sentiment, col_date = [], [], [], []
dates = list(review.sort_values('DATE')['DATE'].unique())

SENTIMENT_MAP = {'satisfied': 'satisfied', 'unsatisfied': 'unsatisfied'}

for sub in TARGET_SUBBRANDS:
    for date_ in dates:
        rev = review[(review['SUBBRAND'] == sub) & (review['DATE'] == date_)]
        for i in range(int((len(rev) + 2) / 3)):
            try:
                str_d = extract_keywords(rev[['Rate', 'Text']][i*3:(i+1)*3].values, sub)
                d = ast.literal_eval(str_d)
                for sentiment_key, col_sentiment_name in [('satisfied', 'satisfied'), ('unsatisfied', 'unsatisfied')]:
                    for keyword in d.get(sentiment_key, []):
                        col_sub.append(sub)
                        col_word.append(keyword)
                        col_sentiment.append(col_sentiment_name)
                        col_date.append(date_)
                print(f'Processed: {date_} {sub}')
            except Exception as e:
                print(f'Failed: {e}')
                time.sleep(60)

output = pd.DataFrame({
    'DATE': pd.to_datetime(col_date, format='%Y%m%d').astype(str),
    'START_DT': start,
    'END_DT': end,
    'SUBBRAND': col_sub,
    'KEYWORD': col_word,
    'SENTIMENT': col_sentiment
})

In [ ]:
# ── TRANSLATE KEYWORDS TO ENGLISH ────────────────────────────────────────

translator = Translator()
translation_map = {}

for keyword in output['KEYWORD'].unique():
    try:
        if keyword:
            translation_map[keyword] = translator.translate(keyword, src='ko', dest='en').text
        time.sleep(0.3)
    except Exception as e:
        print(f'Translation failed for "{keyword}": {e}')

output['KEYWORD_EN'] = output['KEYWORD'].map(translation_map)
output['SENTIMENT_EN'] = output['SENTIMENT']  # already in English

In [ ]:
# ── UPLOAD KEYWORDS TO DATA WAREHOUSE ───────────────────────────────────

engine = create_engine(URL(
    account=os.environ['SNOWFLAKE_ACCOUNT'],
    user=os.environ['SNOWFLAKE_USER'],
    password=os.environ['SNOWFLAKE_PASSWORD'],
    database=os.environ['SNOWFLAKE_DATABASE'],
    schema=os.environ['SNOWFLAKE_SCHEMA'],
    warehouse=os.environ['SNOWFLAKE_WAREHOUSE'],
))

with engine.connect() as con:
    output.to_sql(name='KEYWORDS_TABLE', con=con, if_exists='append',
                  method=pd_writer, index=False)

print(f'Uploaded {len(output)} keyword records')

## Part 2: Property Rating + Weekly Summary
Aggregates keywords into property-level scores (1-10) and generates a 2-sentence consumer summary per sub-brand.

In [ ]:
# ── LOAD KEYWORD DATA ────────────────────────────────────────────────────

df_query = 'SELECT * FROM KEYWORDS_TABLE'
df = pd.DataFrame(cur.execute(df_query))
df = df.rename(columns={1: 'date', 2: 'subbrand', 3: 'keyword', 4: 'sentiment'})
df['date'] = pd.to_datetime(df['date'], format='%Y-%m-%d').astype(str)

In [ ]:
# ── GPT-4 SCORING AND SUMMARY ────────────────────────────────────────────

PROPERTIES = ['Absorbency', 'Softness', 'Comfort', 'Price', 'Hypoallergenic']
PROPERTIES_KO = ['흡수력', '부드러움', '편함', '가격', '피부 저자극']  # Korean labels for prompt context
PROPERTY_TRANSLATION = dict(zip(PROPERTIES_KO, PROPERTIES))

def make_score(pros, cons):
    """Generate 5 property scores (1-10) based on extracted keywords."""
    prompt = (
        f'Pros: {pros}\nCons: {cons}\n'
        f'Based on these review keywords, score each of {PROPERTIES_KO} out of 10. '
        'Return only a list of 5 scores in order. No other output.'
    )
    messages = [
        {'role': 'system', 'content': 'You are a marketer for the Huggies diaper brand.'},
        {'role': 'user', 'content': prompt}
    ]
    response = openai.ChatCompletion.create(model=GPT_MODEL, messages=messages)
    return response.choices[0].message.content

def make_summary(pros, cons):
    """Generate a 2-sentence consumer opinion summary in Korean."""
    prompt = (
        f'Pros: {pros}\nCons: {cons}\n'
        'Summarize consumer opinion of this product in 2 sentences or less in Korean, '
        'based on keyword frequency and product characteristics.'
    )
    messages = [
        {'role': 'system', 'content': 'You are a marketer for the Huggies diaper brand.'},
        {'role': 'user', 'content': prompt}
    ]
    response = openai.ChatCompletion.create(model=GPT_MODEL, messages=messages)
    return response.choices[0].message.content

In [ ]:
# ── GENERATE SCORES AND SUMMARIES ────────────────────────────────────────

rating_output = pd.DataFrame()

for sub in TARGET_SUBBRANDS:
    target_df = df[(df['date'] >= start) & (df['date'] <= end) & (df['subbrand'] == sub)]
    pros = target_df[target_df['sentiment'] == 'satisfied']['keyword'].tolist()
    cons = target_df[target_df['sentiment'] == 'unsatisfied']['keyword'].tolist()

    summary_text = make_summary(pros, cons).replace("'", '').replace('"', '')
    scores_raw = make_score(pros, cons)
    scores = list(map(float, scores_raw.replace('[','').replace(']','').replace("'",'').split(',')))

    print(f'{start} - {end} | {sub} | {summary_text}')

    for i, prop in enumerate(PROPERTIES_KO):
        score_val = scores[i] if scores[i] != 0 else sum(scores) / len(scores)
        rating_output = rating_output.append({
            'START_DT': start, 'END_DT': end,
            'SUBBRAND': sub,
            'PROPERTY': prop,
            'PROPERTY_EN': PROPERTY_TRANSLATION[prop],
            'RATING': score_val,
            'SUMMARY': summary_text
        }, ignore_index=True)

rating_output = rating_output.sort_values('START_DT')

In [ ]:
# ── TRANSLATE SUMMARIES AND UPLOAD ───────────────────────────────────────

summary_translation = {}
for s in rating_output['SUMMARY'].unique():
    try:
        if s:
            summary_translation[s] = translator.translate(s, src='ko', dest='en').text
        time.sleep(0.3)
    except Exception as e:
        print(f'Translation failed: {e}')

rating_output['SUMMARY_EN'] = rating_output['SUMMARY'].map(summary_translation)

with engine.connect() as con:
    rating_output.to_sql(name='RATING_SUMMARY_TABLE', con=con, if_exists='append',
                         method=pd_writer, index=False)

print(f'Uploaded {len(rating_output)} rating records')

## Part 3: Risk Keyword Detection
Scans all reviews for mentions of hazardous materials (foreign objects, mold, chemicals, etc.).

In [ ]:
# ── RISK KEYWORD DETECTION ───────────────────────────────────────────────
# Flags reviews mentioning contamination or safety-related keywords

RISK_KEYWORDS = [
    'HazardousSubstance', 'Carcinogen', 'VOC', 'Benzene', 'Toluene',
    'Styrene', 'Xylene', 'ToxicSubstance', 'Trichloroethylene',
    'Hazardous', 'Ethylbenzene', 'TVOC', 'ForeignObject',
    'RustWater', 'Bond', 'Mold', 'Bug', 'MetalParticle', 'Metal'
]

# Korean source keywords mapped to English labels
RISK_KEYWORDS_KO = [
    '유해물질','발암','휘발성유기화합물','VOC','벤젠','톨루엔',
    '스티렌','자일렌','독성물질','트리클로로에틸렌','유해성','에틸벤젠',
    '총휘발성유기화합물','TVOC','이물질','녹물','본드','곰팡이','벌레','쇳가루','금속'
]

risk_review_query = f"""
SELECT DATE, max(BRAND), max(SUBBRAND), REVIEW
FROM REVIEW_TABLE
WHERE DATE >= '{start}' AND DATE <= '{end}'
GROUP BY DATE, REVIEW;
"""
risk_review = pd.DataFrame(cur.execute(risk_review_query))
risk_review.columns = ['DATE', 'BRAND', 'SUBBRAND', 'REVIEW']

# Detect risk keywords in each review
def detect_risk(review_text):
    found = []
    for kw in RISK_KEYWORDS_KO:
        if kw in review_text:
            found.append(kw)
    return ', '.join(found)

risk_review['RISK_KEYWORD'] = risk_review['REVIEW'].apply(detect_risk)

# Translate reviews to English
review_translation = {}
for text in risk_review['REVIEW'].unique():
    try:
        if text:
            review_translation[text] = translator.translate(text, src='ko', dest='en').text
        time.sleep(0.3)
    except Exception as e:
        print(f'Translation failed: {e}')

# Translate risk keywords to English
risk_review['RISK_KEYWORD_EN'] = risk_review['RISK_KEYWORD'].apply(
    lambda x: translator.translate(x, src='ko', dest='en').text if x else ''
)
risk_review['REVIEW_EN'] = risk_review['REVIEW'].apply(
    lambda x: review_translation.get(x, 'Translation unavailable')
)
risk_review['DATE'] = pd.to_datetime(risk_review['DATE'], format='%Y%m%d').astype(str)
risk_review['START_DT'] = start
risk_review['END_DT'] = end
risk_output = risk_review.sort_values('DATE')

In [ ]:
# ── UPLOAD RISK DATA ─────────────────────────────────────────────────────

with engine.connect() as con:
    risk_output.to_sql(name='RISK_TABLE', con=con, if_exists='append',
                       method=pd_writer, index=False)

print(f'Uploaded {len(risk_output)} risk records')